# QICKoDeS Tutorial (tProc v1) — the API you're actually running

You confirmed you're on **tProc v1** firmware, so this notebook covers the API you
actually use: `qickodes.instruments`, `channels`, `parameters`, `instruction_base`,
`protocol_base`, `instructions/`, `protocols/`. **`qickodes_tutorial.ipynb` covers tProc
**v2** instead** -- a parallel, differently-named hierarchy for a different firmware. If
you ever see `macro_list`, `Macro`, `PlayPulse`, `hardware_loop_counts`, or `QickSweep1D`,
that's v2 -- none of those exist in the API this notebook covers. Section 3 below has a
full v1-vs-v2 name-translation table to keep the two straight for good.

This assumes you've read **`test_1.ipynb`** (raw QICK). Same principle as the v2 notebook:
every important qickodes class turns out to be a thin wrapper around a method you already
know -- `declare_gen`, `declare_readout`, `set_pulse_registers`, `add_gauss`, `pulse()`,
`trigger()`, `NDAveragerProgram`, `QickSweep`/`add_sweep`. Every section below points out
that connection explicitly, quoting the actual source.

1. Quick QCoDeS recap (full primer is in `qickodes_tutorial.ipynb` section 2)
2. `QickInstrument` (v1): structure and connecting to the board
3. v1 vs v2: a full name-translation table
4. `DacChannel`/`AdcChannel` -- traced back to `declare_gen`/`declare_readout`
5. Hardware-aware Parameters (`HzParameter`, `GainParameter`, ...) -- traced back to `freq2reg`/`get_maxv`
6. `QickInstruction` -- the base "step" class, worked through 3 real examples
7. `SweepProtocol`/`SweepProgram` -- traced back to `NDAveragerProgram`
8. Named protocols (`S21Protocol`, and what else exists)
9. `SoftwareSweep` vs `HardwareSweep`, including the loop-ordering gotcha
10. Full worked example (adapted from the real `example_scripts/`)
11. "Can I define my own program?" -- four levels of control
12. Cheat sheet + glossary + where to go next

**Convention, same as before:** ⚠️ = requires hardware + qcodes/qickodes installed, not
executed here. ✅ = plain Python, actually executed in this notebook.


## 1. Quick QCoDeS recap

If you haven't read `qickodes_tutorial.ipynb` section 2, go read it first -- it covers
`Parameter`, `Instrument`/`InstrumentChannel`/`InstrumentModule`, `Station`, `Measurement`,
and the database/snapshot system in full, with the "universal remote + self-writing
logbook" analogy. Nothing in that primer is v1/v2-specific; it all applies unchanged here.


## 2. `QickInstrument` (v1): structure and connection

Checking `instruments.py`, `QickInstrument.__init__` does the same Pyro4 connection as v2
(and the same as `test_1.ipynb` section 4):
```python
self.soc, self.soccfg = make_proxy(ns_host, ns_port)
```
then explicitly checks the firmware and records it:
```python
tproc_type = self.soccfg["tprocs"][0]["type"]
if tproc_type == "axis_tproc64x32_x8":
    tproc_version = 1
else:
    raise NotImplementedError(f"unsupported tProc type: {tproc_type}")
self.tproc_version = Parameter(..., initial_cache_value=tproc_version)
```
This `tproc_version` parameter is what you'll see asserted (`assert parent.tproc_version.get() == 1`)
all over `protocol_base.py`/`instruction_base.py` -- it's how this whole API layer refuses
to run against v2 firmware by mistake.

**Submodules**: `qi.dacs`, `qi.adcs`, `qi.ddr4_buffer` -- same three as v2. **No
`macro_list`** -- that's v2-only (section 3's table).

**A nice extra**: `cycles2sec_tproc`/`sec2cycles_tproc` -- direct wrappers around
`soccfg.cycles2us`/`soccfg.us2cycles` (`test_1.ipynb` section 1), used by
`TProcSecParameter` (section 5) for any timing value expressed in the tProc's own clock
rather than a specific DAC/ADC channel's clock.

```python
from qickodes import QickInstrument
qi = QickInstrument("ip.address.of.board", ns_port=8888, name="qi")
print(qi.soccfg)
```


## 3. v1 vs v2: full name translation

This is the table to resolve the mix-up from your notes. Same jobs, different classes,
different files -- never mix these two columns:

| Concept | v1 (this notebook) | v2 (`qickodes_tutorial.ipynb`) |
|---|---|---|
| Top-level import | `from qickodes import QickInstrument, HardwareSweep, SoftwareSweep` -- **this is the correct, intended import for you**; no gotcha | `from qickodes.instrument_v2 import QickInstrument, SoftwareSweep` (top-level import resolves to v1!) |
| The whole board | `QickInstrument` (`instruments.py`) | `QickInstrument` (`instrument_v2.py`) -- same name, different file |
| DAC/ADC channel | `DacChannel`/`AdcChannel` (`channels.py`) | `DacChannel`/`AdcChannel` (`channels_v2.py`) |
| One step in a sequence | `QickInstruction` (`instruction_base.py`) | `Macro` (`macro_base_v2.py`) |
| A concrete pulse type | `ConstantPulse`/`GaussianPulse`/... (`instructions/`) | `ConstantPulse`/`ArbitraryPulse` (`pulses_v2/`) + separate `DacEnvelope` classes |
| Where the sequence lives | a plain Python list, passed to `SimpleSweepProtocol(qi, [...])` -- **not stored on `qi`** | `qi.macro_list`, a real submodule on the instrument |
| Orchestrating a run | a separate `SweepProtocol` object's `.run(...)` | `qi.run(...)` directly |
| Underlying tProc-v1 class | `NDAveragerProgram` (`qick.averager_program`) | `AveragerProgramV2` (`qick.asm_v2`) |
| Hardware-sweepable value type | `HardwareParameter` (`parameters.py`) -- `float2int`/`int2float` via `get_parser`/`set_parser` | `SweepableParameter` (`parameters_v2.py`) -- holds a `QickParam` |
| Declaring a hardware sweep | `HardwareSweep(param, start, stop, num)`, passed in a list to `.run(hardware_sweeps=[...])` | `param.set(QickSweep1D("name", start, stop))` + `hardware_loop_counts={"name": num}` |
| Software sweep | `SoftwareSweep` (`protocol_base.py`) | `SoftwareSweep` (`instrument_v2.py`) -- same name, nearly identical, different file |


## 4. `DacChannel`/`AdcChannel` -- traced back to `declare_gen`/`declare_readout`

Checking `channels.py`, this is word-for-word the same connection as v2 (and as
`test_1.ipynb` sections 5/7):

```python
class DacChannel(InstrumentChannel):
    def initialize(self, program):
        program.declare_gen(ch=self.channel_num, nqz=self.nqz.get())

class AdcChannel(InstrumentChannel):
    def initialize(self, program):
        program.declare_readout(ch=self.channel_num, freq=self.freq.get() / 1e6,
                                 phase=0, length=self.length.get_raw(),
                                 gen_ch=self.matching_dac.get())
```

Usage:
```python
readout_dac = qi.dacs[0]
readout_adc = qi.adcs[0]
readout_dac.matching_adc.set(readout_adc.channel_num)   # gen_ch=/ro_ch= frequency matching
readout_adc.matching_dac.set(readout_dac.channel_num)   # from test_1.ipynb sections 5, 7
readout_dac.nqz.set(2)
```

Also worth knowing: `DacChannel`/`AdcChannel` each carry `hz2reg`/`reg2hz`, `deg2reg`/`reg2deg`,
`sec2cycles`/`cycles2sec` helper methods -- thin wrappers around `soccfg.freq2reg`/
`reg2freq`, `deg2reg`/`reg2deg`, `us2cycles`/`cycles2us` (`test_1.ipynb` section 1). These
are what section 5's Parameters use internally.


## 5. Hardware-aware Parameters -- traced back to `freq2reg`/`get_maxv`

This is the piece that's genuinely different in spirit from v2's `SweepableParameter`
(which stores an optional `QickParam` object). v1 instead uses a real, plain QCoDeS
mechanism: `Parameter(get_parser=..., set_parser=...)` -- a function that transforms every
value automatically on `.get()`/`.set()`. Checking `parameters.py`:

```python
class HardwareParameter(ABC, ManualParameter):
    def __init__(self, name, **kwargs):
        super().__init__(name, get_parser=self.int2float, set_parser=self.float2int, **kwargs)
```

Each subclass just plugs in the conversion you already know from `test_1.ipynb` section 1:

| Class | `float2int` wraps | `int2float` wraps |
|---|---|---|
| `HzParameter` | `channel.hz2reg` -> `soccfg.freq2reg` | `channel.reg2hz` -> `soccfg.reg2freq` |
| `DegParameter` | `soccfg.deg2reg` | `soccfg.reg2deg` |
| `SecParameter` | `channel.sec2cycles` -> `soccfg.us2cycles` (per-channel clock) | `channel.cycles2sec` -> `soccfg.cycles2us` |
| `TProcSecParameter` | `qi.sec2cycles_tproc` -> `soccfg.us2cycles` (tProc clock) | `qi.cycles2sec_tproc` |
| `GainParameter` | `round(f * 32768)` | `i / 32768` |

`GainParameter`'s formula is worth pausing on: **32768 = 2^15** -- the exact DAC full-scale
code range from `test_1.ipynb` section 6.2/9 (`get_maxv`, "gain code ~ +-32000"), just
hardcoded here instead of read from `get_maxv(ch)` per channel. So `gain.set(0.72)` stores
`round(0.72 * 32768)` and hands *that* integer to QICK -- the `freq2reg`/`get_maxv`
rounding theme from `test_1.ipynb`, fully automated, invisible on every `.set()`/`.get()`.


## 6. `QickInstruction` -- the base "step" class

The v1 equivalent of v2's `Macro`. Every instruction implements three methods, and every
one of them just calls a `test_1.ipynb` method you already know. Three real examples,
traced from `instructions/`:

### `ConstantPulse` (`instructions/constant_pulse.py`)
```python
def initialize(self, program):
    program.set_pulse_registers(ch=..., style="const", freq=self.freq.get_raw(),
                                 phase=0, gain=self.gain.get_raw(), length=self.length.get_raw(),
                                 phrst=0, stdysel="zero", mode="oneshot")
    # <- test_1.ipynb section 6, verbatim

def append_to(self, program):
    program.pulse(ch=self.dacs[0].channel_num, t="auto")   # <- test_1.ipynb section 6, verbatim

def add_sweep(self, program, sweep):
    reg = program.get_gen_reg(self.dacs[0].channel_num, "gain")
    program.add_sweep(QickSweep(program, reg, sweep.start_int, sweep.stop_int, sweep.num))
    # <- test_1.ipynb section 8.1's NDSweepProgram pattern, verbatim
```

### `GaussianPulse` (`instructions/gaussian_pulse.py`) -- the `"arb"` style
```python
def initialize(self, program):
    program.add_gauss(ch=..., name=self.name, sigma=self.sigma.get_raw(), length=self.length.get_raw())
    program.set_pulse_registers(ch=..., style="arb", freq=..., gain=..., waveform=self.name, ...)
    # <- test_1.ipynb sections 6.1/6, verbatim
```

### `Readout` (`instructions/readout.py`) -- composes another instruction
```python
def initialize(self, program):
    self.pulse.initialize(program)   # delegates to whatever pulse it was built with
    self.wait_before_reg = program.new_gen_reg(gen_ch=..., init_val=..., reg_type="time", tproc_reg=True)
    # <- test_1.ipynb section 8.1's QickRegisterManagerMixin.new_gen_reg, verbatim

def append_to(self, program):
    program.sync_all()
    program.sync(self.wait_before_reg.page, self.wait_before_reg.addr)
    program.trigger(adcs=[self.adcs[0].channel_num], adc_trig_offset=self.adc_trig_offset.get_raw())
    # <- test_1.ipynb section 7's trigger(), verbatim
    self.pulse.append_to(program)          # plays the wrapped pulse
    if self.wait_for_adc.get():
        program.wait_all()                  # <- test_1.ipynb section 6
    program.sync_all(t=self.wait_after.get_raw())
```
`Readout` is a genuinely useful pattern: it *wraps* a `ConstantPulse`/`GaussianPulse` and
adds the trigger/sync/wait choreography around it, so you build a readout channel once
(pulse + timing) and reuse it, rather than repeating `trigger()`/`sync_all()` calls by hand
in every protocol.


## 7. `SweepProtocol`/`SweepProgram` -- traced back to `NDAveragerProgram`

Checking `protocol_base.py`:

```python
class SweepProgram(NDAveragerProgram):
```

**This is the literal `NDAveragerProgram` from `test_1.ipynb` section 8.1**, not a
reimplementation. `SweepProgram.initialize()`:
```python
def initialize(self):
    for dac in self.dacs: dac.initialize(self)          # section 4
    for adc in self.adcs: adc.initialize(self)           # section 4
    for instruction in set(self.protocol.instructions):
        instruction.initialize(self)                      # section 6
    for sweep in reversed(self.hardware_sweeps):
        sweep.parameter.instrument.add_sweep(self, sweep)  # section 6's add_sweep()
    self.synci(200)   # <- test_1.ipynb section 3.1's ASM instruction, verbatim
```

`SweepProtocol.run()` mirrors v2's `QickInstrument.run()`: register sweep/result parameters
with the `Measurement`, generate the program (`self.generate_program(...)`, abstract --
subclasses decide how), loop over software sweep combinations if any, and for each, call
`run_hardware_sweeps()`/`run_hardware_sweeps_decimated()`, which build the program and call
its own `.acquire()`/`.acquire_decimated()`:

```python
all_iq = program.acquire(soc=self.parent.soc, load_pulses=True, progress=progress)
```

Note this calls `program.acquire()` (the program's *own* method, ultimately
`NDAveragerProgram.acquire()` -> `AcquireMixin.acquire()` from `test_1.ipynb` section 7.1)
rather than `AcquireMixin.acquire()` directly -- a documented, deliberate choice, per the
comment in the source: the program-level method also handles `set_reads_per_shot()` and
`save_experiments` bookkeeping that calling `AcquireMixin.acquire()` directly would skip.
Underneath that, though, it's the exact same circular-buffer/background-thread pipeline
from section 7.1 -- unchanged.

One more real detail worth knowing: `SweepProgram` overrides `_summarize_accumulated`/
`_summarize_decimated` to bypass `NDAveragerProgram`'s own summarizing (which returns the
`(expt_pts, avg_di, avg_dq)` triplet you saw in the very first full `averager_program.py`
paste) in favor of the more generic `AcquireMixin` format the rest of this driver expects
-- a real, working example of "override one method of a class you already understand
instead of reimplementing the whole thing."


## 8. Named protocols -- pre-built recipes on top of `SimpleSweepProtocol`

`SimpleSweepProtocol`/`SimpleSweepProgram` are the plainest concrete implementation:
```python
class SimpleSweepProgram(SweepProgram):
    def body(self):
        for instruction in self.protocol.instructions:
            instruction.append_to(self)
```
You hand it a list of instructions (section 6), it plays them in order -- the v1 sibling of
v2's `macro_list`.

`protocols/` has a few ready-made ones built the same way. The simplest, `S21Protocol`
(`protocols/s21.py`), is genuinely just this:
```python
class S21Protocol(SimpleSweepProtocol):
    def __init__(self, parent, readout, name="S21Protocol", **kwargs):
        super().__init__(parent=parent, instructions=[readout], name=name, **kwargs)
```
-- a `SimpleSweepProtocol` pre-loaded with exactly one `Readout` instruction (section 6),
nothing more. `protocols/pulse_probe.py`, `ramsey.py`, `hahn_echo.py` follow the same idea
with more instructions in their fixed sequence (e.g. `PulseProbeProtocol`, used in the
`meas_pulse_probe_2d.py` file you had open earlier, adds a qubit-drive pulse before the
readout).


## 9. `SoftwareSweep` vs `HardwareSweep`

Same decision rule as ever: register-only quantity -> `HardwareSweep`, hardware-executed,
nanoseconds between points (built on `NDAveragerProgram`/`add_sweep`, section 7). Anything
needing a fresh program/envelope reload -> `SoftwareSweep`, a Python loop, milliseconds
between points.

```python
p.run(
    Measurement(experiment, station, name),
    software_sweeps=[SoftwareSweep(qubit_pulse.gain, [0.0009, 0.009, 0.09, 0.9])],
    hardware_sweeps=[HardwareSweep(qubit_pulse.freq, 2.7e9, 3.0e9, 601)],
)
```

**The ordering gotcha**, traced through `protocol_base.py`'s `SweepProgram.initialize()`:
```python
for sweep in reversed(self.hardware_sweeps):
    sweep.parameter.instrument.add_sweep(self, sweep)
```
Per `test_1.ipynb` section 8.1, the *first* sweep added to `NDAveragerProgram` becomes the
*innermost* loop. Because this code iterates `hardware_sweeps` **reversed**, the *first*
entry in the list you write becomes the **outermost** loop, and the *last* entry becomes
the **innermost**. Reading your `hardware_sweeps=[...]` list top-to-bottom corresponds to
outer-to-inner nesting -- an intuitive convention, but only because of that `reversed()`
call, so it's worth knowing it's there.


## 10. Full worked example -- S21 vs frequency

Directly adapted from the real `example_scripts/header.py` + `meas_resonator.py` in this
repo -- the README's own recommended first example.


In [ ]:
# ⚠️ requires hardware + qcodes/qickodes installed -- S21 vs frequency, tProc v1.
import os
from qcodes import Measurement, Station, initialise_or_create_database_at, load_or_create_experiment
from qickodes import QickInstrument, SoftwareSweep
from qickodes.instructions import ConstantPulse, Readout
from qickodes.protocols import S21Protocol

experiment_name = "loopback"
sample_name = "none"
wiring = ["ZCU216-DAC0_230 - balun_5-6GHz - balun_5-6GHz - ZCU216-ADC0_226"]

initialise_or_create_database_at(f"./database/{experiment_name}.db")
experiment = load_or_create_experiment(experiment_name, sample_name)

station = Station()
station.metadata["wiring"] = wiring
qick_instrument = QickInstrument("10.0.100.16")   # your board's IP
station.add_component(qick_instrument)

readout_dac = qick_instrument.dacs[0]
readout_adc = qick_instrument.adcs[0]
readout_dac.matching_adc.set(readout_adc.channel_num)
readout_adc.matching_dac.set(readout_dac.channel_num)
readout_dac.nqz.set(2)

readout_pulse = ConstantPulse(qick_instrument, readout_dac, "readout_pulse")
readout_pulse.gain.set(0.02)
readout_pulse.freq.set(6.3203e9)
readout_pulse.length.set(10e-6)
readout_adc.freq.set(readout_pulse.freq.get())
readout_adc.length.set(readout_pulse.length.get())

readout = Readout(qick_instrument, readout_pulse, readout_adc)
readout.wait_before.set(0)
readout.wait_after.set(10e-6)
readout.adc_trig_offset.set(0.5e-6)

p = S21Protocol(qick_instrument, readout)
p.hard_avgs.set(1000)

run_id = p.run(
    Measurement(experiment, station, "meas_resonator"),
    software_sweeps=[
        SoftwareSweep([readout_pulse.freq, readout_adc.freq], 6.317e9, 6.323e9, 601),
    ],
)
print(f"run_id = {run_id}")


## 11. "Can I define my own program?" -- four levels

Yes -- and every level below is exactly the same raw-QICK code you already know from
`test_1.ipynb`, just wrapped a little differently each time.

1. **No new classes**: `SimpleSweepProtocol(qi, [instruction1, instruction2, ...])` --
   supply your own ordered list of *existing* `instructions/` objects.
2. **Your own instruction type**: subclass `QickInstruction` yourself, exactly like
   `ConstantPulse`/`GaussianPulse`/`Readout` do (section 6) -- write `initialize()`/
   `append_to()`/`add_sweep()` using `set_pulse_registers`/`add_gauss`/`pulse()`/`trigger()`/
   `get_gen_reg`+`QickSweep`, for a pulse shape or step type that doesn't already exist.
3. **Your own protocol**: subclass `SweepProtocol` (like `S21Protocol`/`PulseProbeProtocol`
   do) and implement `generate_program()` yourself for full control over `body()` beyond
   "play a list in order," while still getting `HardwareSweep`/`SoftwareSweep`/dataset
   saving for free.
4. **Skip qickodes' protocol layer entirely**: since `SweepProgram` *is* `NDAveragerProgram`,
   nothing stops you from writing a raw `AveragerProgram`/`RAveragerProgram`/
   `NDAveragerProgram` exactly as in `test_1.ipynb`, with zero qickodes involvement --
   optionally wrapped in `Instrument.add_parameter(get_cmd=...)` (see the "I feel I don't
   have enough control" discussion) if you still want it inside a QCoDeS `Station`.


## 12. Cheat sheet: v1 qickodes -> raw-QICK equivalent (`test_1.ipynb`)

| qickodes (v1) | test_1.ipynb equivalent |
|---|---|
| `QickInstrument(ns_host, ns_port)` | `make_proxy(...)`, section 4 |
| `DacChannel`/`AdcChannel.initialize()` | `declare_gen`/`declare_readout`, sections 5, 7 |
| `HzParameter`/`DegParameter`/`SecParameter`/`GainParameter` | `freq2reg`/`deg2reg`/`us2cycles`/`get_maxv`, section 1 |
| `ConstantPulse.initialize()`/`.append_to()` | `set_pulse_registers`/`pulse()`, section 6 |
| `GaussianPulse.initialize()` | `add_gauss` + `set_pulse_registers(style="arb")`, section 6.1 |
| `Readout.append_to()` | `trigger()`/`sync_all()`/`wait_all()`, sections 6, 7 |
| `SweepProgram` | `NDAveragerProgram`, section 8.1 |
| `SimpleSweepProgram.body()` | hand-written `body()`, section 5 |
| `SweepProtocol.run()` -> `program.acquire()` | `AcquireMixin.acquire()`, section 7.1 |
| `HardwareSweep` + `add_sweep` | `QickSweep` + `add_sweep`, section 8.1 |
| `SoftwareSweep` | the Python `for` loop pattern, section 6.2 |


## 13. Where to go next

- **In this repo**: `src/libraries/qickodes/example_scripts/` has the rest of the README's
  recommended examples -- `meas_s21_vs_adc_trig_offset.py`, `meas_punchout.py`,
  `meas_pulse_probe_2d.py`, `meas_pi_pulse_gain_sweep.py`,
  `meas_resonator_vs_qubit_state.py`, `meas_t1.py` -- all following the same
  `header.py` + `Protocol(...)` + `.run(...)` pattern covered here.
- **`src/libraries/qickodes/src/qickodes/protocols/`** -- the other pre-built protocols:
  `pulse_probe.py`, `ramsey.py`, `hahn_echo.py`.
- **`src/libraries/qickodes/src/qickodes/instructions/`** -- the other pre-built
  instructions: `iq_constant_pulse.py`, `delay.py`, `set_phase.py`.
- **`test_1.ipynb`** -- every "traced back to..." callback above assumes you've read it;
  that's the place to revisit if any of them weren't clear.
- **`qickodes_tutorial.ipynb`** -- the v2 sibling of this notebook, useful if your group
  ever migrates this board to tProc v2 firmware.
